# Unified Model Evaluation Pipeline

This notebook evaluates all saved model families with one pipeline using the Section 9+ evaluation structure from the incremental-monthly SGD notebook.

Key policies implemented:
- Incremental-scaler families use the model-year scaler when evaluating prior years.
- Standard-scaler families use the evaluation-year scaler.
- Feature preparation is routed by model family to the original notebook-style function.
- Each evaluation table is saved immediately and supports resume (existing table files are skipped and loaded).
- Final-model evaluation supports an optional final scaler when a family saves one separately.

In [1]:
import json
import pickle
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

ROOT = Path('.')
BASELINE_REFERENCE_NOTEBOOK = ROOT / 'SGD Classifier.ipynb'
EVAL_DIR = ROOT / 'eval_outputs' / 'unified_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATH = ROOT / 'data_split.npz'
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f'Missing split file: {SPLIT_PATH.resolve()}')

split = np.load(SPLIT_PATH)

def _first_existing_key(candidates):
    for key in candidates:
        if key in split.files:
            return key
    return None

train_key = _first_existing_key(['train_pixel_indices', 'train_idx', 'train_indices'])
val_key = _first_existing_key(['val_pixel_indices', 'valid_pixel_indices', 'val_idx', 'val_indices'])
test_key = _first_existing_key(['test_pixel_indices', 'test_idx', 'test_indices'])

if not all([train_key, val_key, test_key]):
    raise KeyError(f'Could not resolve split keys from: {split.files}')

train_pixel_indices = split[train_key]
val_pixel_indices = split[val_key]
test_pixel_indices = split[test_key]

print(f'Loaded splits from {SPLIT_PATH.name}')
print(f'  train: {len(train_pixel_indices):,} | val: {len(val_pixel_indices):,} | test: {len(test_pixel_indices):,}')

FAMILIES = {
    'baseline': {
        'label': 'SGD Baseline',
        'models_dir': ROOT / 'models',
        'model_template': 'model_year_{year}.pkl',
        'scaler_template': 'scaler_year_{year}.pkl',
        'dataset_path': ROOT / 'training_data_with_features.zarr',
        'prep_kind': 'baseline_notebook',
        'reference_notebook': BASELINE_REFERENCE_NOTEBOOK,
        'incremental_scaler': False,
    },
    'huber': {
        'label': 'SGD Huber',
        'models_dir': ROOT / 'models_huber',
        'model_template': 'model_year_{year}_huber.pkl',
        'scaler_template': 'scaler_year_{year}_huber.pkl',
        'dataset_path': ROOT / 'training_data_with_features.zarr',
        'prep_kind': 'prevyears',
        'incremental_scaler': False,
    },
    'lagged_features': {
        'label': 'SGD Lagged Features',
        'models_dir': ROOT / 'models_lagged_features',
        'model_template': 'model_year_{year}_lagged_features.pkl',
        'scaler_template': 'scaler_year_{year}_lagged_features.pkl',
        'dataset_path': ROOT / 'training_data_with_features.zarr',
        'prep_kind': 'prevyears',
        'incremental_scaler': False,
    },
    'lagged_features_incremental_scaler': {
        'label': 'SGD Lagged Features (Incremental Scaler)',
        'models_dir': ROOT / 'models_lagged_features_incremental_scaler',
        'model_template': 'model_year_{year}_lagged_features_incremental_scaler.pkl',
        'scaler_template': 'scaler_year_{year}_lagged_features_incremental_scaler.pkl',
        'dataset_path': ROOT / 'training_data_with_features.zarr',
        'prep_kind': 'prevyears',
        'incremental_scaler': True,
    },
    'neighbours': {
        'label': 'SGD Neighbours',
        'models_dir': ROOT / 'models_neighbours',
        'model_template': 'model_year_{year}.pkl',
        'scaler_template': 'scaler_year_{year}.pkl',
        'dataset_path': ROOT / 'training_data_with_neighbourhood_features.zarr',
        'prep_kind': 'neighbourhood',
        'incremental_scaler': False,
    },
    'prevyears_monthly_features': {
        'label': 'SGD Prevyears + Monthly',
        'models_dir': ROOT / 'models_prevyears_monthly_features',
        'model_template': 'model_year_{year}_prevyears_monthly_features.pkl',
        'scaler_template': 'scaler_year_{year}_prevyears_monthly_features.pkl',
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': False,
    },
    'prevyears_monthly_features_experience_replay': {
        'label': 'SGD Prevyears + Monthly + Replay',
        'models_dir': ROOT / 'models_prevyears_monthly_features_experience_replay',
        'model_template': 'model_year_{year}_prevyears_monthly_features_experience_replay.pkl',
        'scaler_template': 'scaler_year_{year}_prevyears_monthly_features_experience_replay.pkl',
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': False,
        'final_model_path': ROOT / 'sgd_classifier_model_prevyears_monthly_features_experience_replay.pkl',
    },
    'prevyears_monthly_features_incremental_scaler': {
        'label': 'SGD Prevyears + Monthly (Incremental Scaler)',
        'models_dir': ROOT / 'models_prevyears_monthly_features_incremental_scaler',
        'model_template': 'model_year_{year}_prevyears_monthly_features_incremental_scaler.pkl',
        'scaler_template': 'scaler_year_{year}_prevyears_monthly_features_incremental_scaler.pkl',
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': True,
        'final_model_path': ROOT / 'sgd_classifier_model_prevyears_monthly_features_incremental_scaler.pkl',
    },
    'prevyears_monthly_features_incremental_scaler_experience_replay': {
        'label': 'SGD Prevyears + Monthly (Incremental Scaler + Replay)',
        'models_dir': ROOT / 'models_prevyears_monthly_features_incremental_scaler_experience_replay',
        'model_template': 'model_year_{year}_prevyears_monthly_features_incremental_scaler_experience_replay.pkl',
        'scaler_template': 'scaler_year_{year}_prevyears_monthly_features_incremental_scaler_experience_replay.pkl',
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': True,
        'final_model_path': ROOT / 'sgd_classifier_model_prevyears_monthly_features_incremental_scaler_experience_replay.pkl',
    },
    'prevyears_monthly_features_rbf': {
        'label': 'SGD Prevyears + Monthly (RBF)',
        'models_dir': ROOT / 'models_prevyears_monthly_features_rbf',
        'model_template': 'model_year_{year}_rbf.pkl',
        'scaler_template': None,
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': False,
        'rbf_sampler_path': ROOT / 'models_prevyears_monthly_features_rbf' / 'rbf_sampler.pkl',
        'final_model_path': ROOT / 'sgd_classifier_model_prevyears_monthly_features_rbf.pkl',
    },
    'mlp_prevyears_monthly_features_incremental_scaler': {
        'label': 'MLP Prevyears + Monthly (Incremental Scaler)',
        'models_dir': ROOT / 'models_mlp_prevyears_monthly_features_incremental_scaler',
        'model_template': 'model_year_{year}_mlp_prevyears_monthly_features_incremental_scaler.pkl',
        'scaler_template': 'scaler_year_{year}_mlp_prevyears_monthly_features_incremental_scaler.pkl',
        'dataset_path': ROOT / 'training_data_with_features_plus_monthly_indices.zarr',
        'prep_kind': 'monthly',
        'incremental_scaler': True,
        'final_model_path': ROOT / 'mlp_classifier_model_prevyears_monthly_features_incremental_scaler.pkl',
        'final_scaler_path': ROOT / 'models_mlp_prevyears_monthly_features_incremental_scaler' / 'scaler_final_mlp_prevyears_monthly_features_incremental_scaler.pkl',
    },
}

DATASET_CACHE = {}


def load_family_dataset(family_cfg):
    path = family_cfg['dataset_path']
    if path not in DATASET_CACHE:
        if not path.exists():
            raise FileNotFoundError(f'Missing dataset for family: {path.resolve()}')
        DATASET_CACHE[path] = xr.open_zarr(path)
    return DATASET_CACHE[path]

print(f'Configured {len(FAMILIES)} model families')

Loaded splits from data_split.npz
  train: 5,597,776 | val: 1,273,437 | test: 1,283,992
Configured 11 model families


In [2]:
def _prepare_features_common(
    ds,
    pixel_indices,
    year_idx,
    scaler=None,
    scaler_mode='auto',
    include_last_year=True,
    include_monthly=False,
    include_neighbourhood=False,
):
    if year_idx == 0:
        return np.empty((0, 0)), np.empty((0,)), scaler

    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f'Invalid scaler_mode: {scaler_mode}')

    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    features_list = [s2_features, dem_features, ndvi_features, ndwi_features]

    if include_last_year:
        ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
        ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
        ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))
        features_list.extend([ndvi_last_year, ndwi_last_year])

        required_last_year_bands = ['B04', 'B03', 'B06']
        band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
        missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
        if missing_bands:
            raise ValueError(f'Missing required S2 bands for last-year features: {missing_bands}')
        for band in required_last_year_bands:
            band_values = ds_prev['s2_bands'].sel(s2_band=band).values
            band_values = np.where(np.isnan(band_values), 0, band_values)
            features_list.append(band_values.reshape(-1, 1))

    # Keep feature order aligned with training notebooks: append NBR after last-year features.
    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if include_neighbourhood:
        for var_name in ['ndvi_neighbour', 'ndwi_neighbour', 's2_b03_neighbour', 's2_b04_neighbour']:
            if var_name in ds.data_vars:
                features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    if include_monthly:
        yearly_index_feature_names = [
            'ndvi_cv_year',
            'ndvi_max_m2m_drop_year',
            'ndvi_max_year',
            'ndvi_min_year',
            'ndvi_std_year',
            'ndwi_cv_year',
            'ndwi_max_m2m_drop_year',
            'ndwi_max_year',
            'ndwi_min_year',
            'ndwi_std_year',
        ]
        for feature_name in yearly_index_feature_names:
            if feature_name in ds.data_vars:
                features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X = X[nan_mask]
    y = y[nan_mask]

    if len(X) == 0:
        return np.empty((0, 0)), np.empty((0,)), scaler

    if scaler_mode == 'none':
        return X, y, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'n_features_in_'):
            X = scaler.transform(X)
        else:
            X = scaler.fit_transform(X)
    elif scaler_mode == 'fit':
        X = scaler.fit_transform(X)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X)
        X = scaler.transform(X)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'n_features_in_'):
            raise ValueError("Scaler must be fitted before scaler_mode='transform'.")
        X = scaler.transform(X)

    return X, y, scaler


NOTEBOOK_PREP_FN_CACHE = {}


def load_notebook_prepare_features(notebook_path, function_name='prepare_features_for_year'):
    notebook_path = Path(notebook_path)
    cache_key = (str(notebook_path.resolve()), function_name)
    if cache_key in NOTEBOOK_PREP_FN_CACHE:
        return NOTEBOOK_PREP_FN_CACHE[cache_key]

    if not notebook_path.exists():
        raise FileNotFoundError(f'Missing reference notebook: {notebook_path.resolve()}')

    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_payload = json.load(f)

    marker = f'def {function_name}('
    matching_sources = []
    for cell in notebook_payload.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        cell_source = ''.join(cell.get('source', []))
        if marker in cell_source:
            matching_sources.append(cell_source)

    if not matching_sources:
        raise RuntimeError(f'Could not locate {function_name} in {notebook_path}')

    namespace = {
        'np': np,
        'StandardScaler': StandardScaler,
    }
    exec(matching_sources[-1], namespace)
    feature_fn = namespace.get(function_name)
    if feature_fn is None:
        raise RuntimeError(f'Failed to load {function_name} from {notebook_path}')

    NOTEBOOK_PREP_FN_CACHE[cache_key] = feature_fn
    return feature_fn


def prepare_features_prevyears(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
    return _prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=True,
        include_monthly=False,
        include_neighbourhood=False,
    )


def prepare_features_baseline_notebook(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
    if scaler_mode not in {'auto', 'fit', 'transform'}:
        raise ValueError(
            "baseline_notebook prep only supports scaler_mode values 'auto', 'fit', and 'transform'."
        )

    notebook_feature_fn = load_notebook_prepare_features(BASELINE_REFERENCE_NOTEBOOK)

    if scaler_mode == 'fit':
        scaler_arg = None
    elif scaler_mode == 'transform':
        if scaler is None:
            raise ValueError("A fitted scaler is required for baseline_notebook with scaler_mode='transform'.")
        scaler_arg = scaler
    else:
        scaler_arg = scaler

    return notebook_feature_fn(ds, pixel_indices, year_idx, scaler=scaler_arg)


def prepare_features_monthly(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
    return _prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=True,
        include_monthly=True,
        include_neighbourhood=False,
    )


def prepare_features_neighbourhood(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
    return _prepare_features_common(
        ds=ds,
        pixel_indices=pixel_indices,
        year_idx=year_idx,
        scaler=scaler,
        scaler_mode=scaler_mode,
        include_last_year=False,
        include_monthly=False,
        include_neighbourhood=True,
    )


PREP_FN_BY_KIND = {
    'baseline_notebook': prepare_features_baseline_notebook,
    'prevyears': prepare_features_prevyears,
    'monthly': prepare_features_monthly,
    'neighbourhood': prepare_features_neighbourhood,
}


def compute_metrics(y_true, y_pred, y_proba):
    out = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': np.nan,
        'pr_auc': np.nan,
    }
    if len(np.unique(y_true)) > 1:
        out['roc_auc'] = roc_auc_score(y_true, y_proba)
        out['pr_auc'] = average_precision_score(y_true, y_proba)
    return out


def best_f1_threshold(y_true, y_score, default_threshold=0.5):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float(default_threshold), {'source': 'fallback', 'reason': 'insufficient classes', 'n_val_samples': int(len(y_true))}

    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return float(default_threshold), {'source': 'fallback', 'reason': 'no thresholds', 'n_val_samples': int(len(y_true))}

    f1_vals = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-12)
    best_idx = int(np.argmax(f1_vals[:-1])) if len(f1_vals) > 1 else 0
    threshold = float(thresholds[min(best_idx, len(thresholds) - 1)])
    return threshold, {
        'source': 'computed_pr_f1',
        'reason': 'max F1 on validation PR curve',
        'n_val_samples': int(len(y_true)),
    }


def save_table(df, csv_path):
    df.to_csv(csv_path, index=False)
    json_path = csv_path.with_suffix('.json')
    payload = {
        'metadata': {
            'generated_at_utc': datetime.now(timezone.utc).isoformat(),
            'row_count': int(len(df)),
            'columns': list(df.columns),
        },
        'rows': df.to_dict(orient='records'),
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)


def load_or_run_table(table_csv_path, fn_compute):
    if table_csv_path.exists():
        print(f'SKIP existing table: {table_csv_path.name}')
        return pd.read_csv(table_csv_path), True
    table_df = fn_compute()
    save_table(table_df, table_csv_path)
    print(f'SAVED table: {table_csv_path.name} ({len(table_df):,} rows)')
    return table_df, False

print('Feature preparation and helper functions loaded')

Feature preparation and helper functions loaded


In [3]:
def infer_years_from_models(models_dir, model_template):
    years = []
    for y in range(2010, 2035):
        if (models_dir / model_template.format(year=y)).exists():
            years.append(y)
    return years


def load_model(models_dir, model_template, year):
    path = models_dir / model_template.format(year=year)
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)


def load_scaler(models_dir, scaler_template, year):
    if scaler_template is None:
        return None
    path = models_dir / scaler_template.format(year=year)
    if not path.exists():
        return None
    with open(path, 'rb') as f:
        return pickle.load(f)


def load_final_scaler(family_cfg):
    final_scaler_path = family_cfg.get('final_scaler_path')
    if not final_scaler_path:
        return None
    if not Path(final_scaler_path).exists():
        return None
    with open(final_scaler_path, 'rb') as f:
        return pickle.load(f)


def resolve_scaler_year(family_cfg, model_year, eval_year):
    if family_cfg['incremental_scaler'] and eval_year < model_year:
        return model_year, 'model_year_scaler_for_prior_year'
    return eval_year, 'evaluation_year_scaler'


def validate_feature_count(X, expected_n_features, owner_label, family_id, model_year, eval_year, split_name):
    if expected_n_features is None:
        return

    n_features = int(X.shape[1])
    expected_n_features = int(expected_n_features)
    if n_features != expected_n_features:
        raise ValueError(
            f'Feature mismatch for family={family_id}, model_year={model_year}, eval_year={eval_year}, split={split_name}: '
            f'X has {n_features} features, but {owner_label} expects {expected_n_features} features.'
        )


def validate_scaler_features(X, scaler_obj, family_id, model_year, eval_year, split_name):
    expected_n_features = getattr(scaler_obj, 'n_features_in_', None) if scaler_obj is not None else None
    validate_feature_count(
        X=X,
        expected_n_features=expected_n_features,
        owner_label='loaded scaler',
        family_id=family_id,
        model_year=model_year,
        eval_year=eval_year,
        split_name=split_name,
    )


def validate_model_features(X, model_obj, family_id, model_year, eval_year, split_name):
    expected_n_features = getattr(model_obj, 'n_features_in_', None) if model_obj is not None else None
    validate_feature_count(
        X=X,
        expected_n_features=expected_n_features,
        owner_label='loaded model',
        family_id=family_id,
        model_year=model_year,
        eval_year=eval_year,
        split_name=split_name,
    )


def compute_threshold_for_pair(model_obj, prep_fn, ds, family_cfg, model_year, eval_year, scaler_obj, rbf_sampler=None, default_threshold=0.5):
    year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
    eval_year_idx = year_to_idx[int(eval_year)]
    if scaler_obj is None:
        val_scale_mode = 'none'
    else:
        val_scale_mode = 'transform'

    X_val, y_val, _ = prep_fn(ds, val_pixel_indices, eval_year_idx, scaler=scaler_obj, scaler_mode=val_scale_mode)
    if len(X_val) == 0:
        return float(default_threshold), {'source': 'fallback', 'reason': 'empty validation set', 'n_val_samples': 0}

    validate_scaler_features(X_val, scaler_obj, family_cfg['label'], model_year, eval_year, 'validation')

    if rbf_sampler is not None:
        X_val = rbf_sampler.transform(X_val)

    validate_model_features(X_val, model_obj, family_cfg['label'], model_year, eval_year, 'validation')

    y_val_proba = model_obj.predict_proba(X_val)[:, 1]
    return best_f1_threshold(y_val, y_val_proba, default_threshold=default_threshold)


def evaluate_one_pair(model_obj, prep_fn, ds, family_cfg, model_year, eval_year, scaler_obj, threshold, threshold_meta, rbf_sampler=None):
    year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
    eval_year_idx = year_to_idx[int(eval_year)]

    if scaler_obj is None:
        scale_mode = 'none'
    else:
        scale_mode = 'transform'

    X_test, y_test, _ = prep_fn(ds, test_pixel_indices, eval_year_idx, scaler=scaler_obj, scaler_mode=scale_mode)
    if len(X_test) == 0:
        return None

    validate_scaler_features(X_test, scaler_obj, family_cfg['label'], model_year, eval_year, 'test')

    if rbf_sampler is not None:
        X_test = rbf_sampler.transform(X_test)

    validate_model_features(X_test, model_obj, family_cfg['label'], model_year, eval_year, 'test')

    y_proba = model_obj.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    metrics = compute_metrics(y_test, y_pred, y_proba)

    return {
        'model_year': int(model_year),
        'eval_year': int(eval_year),
        'n_test_samples': int(len(y_test)),
        'threshold_used': float(threshold),
        'threshold_source': threshold_meta.get('source', 'unknown'),
        'threshold_reason': threshold_meta.get('reason', ''),
        'n_val_threshold_samples': int(threshold_meta.get('n_val_samples', 0)),
        **metrics,
    }


def evaluate_family(family_id, family_cfg, default_threshold=0.5):
    prep_fn = PREP_FN_BY_KIND[family_cfg['prep_kind']]
    ds = load_family_dataset(family_cfg)
    models_dir = family_cfg['models_dir']

    if not models_dir.exists():
        raise FileNotFoundError(f'Model directory missing: {models_dir.resolve()}')

    years = infer_years_from_models(models_dir, family_cfg['model_template'])
    if not years:
        raise RuntimeError(f'No model years found for {family_id}')

    rbf_sampler = None
    if 'rbf_sampler_path' in family_cfg:
        rbf_path = family_cfg['rbf_sampler_path']
        if rbf_path.exists():
            with open(rbf_path, 'rb') as f:
                rbf_sampler = pickle.load(f)

    family_out_dir = EVAL_DIR / family_id
    family_out_dir.mkdir(parents=True, exist_ok=True)

    status_rows = []

    def _run_table_own_year():
        rows = []
        for year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], year)
            if model_obj is None:
                continue
            scaler_year, scaler_policy = resolve_scaler_year(family_cfg, year, year)
            scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

            threshold, threshold_meta = compute_threshold_for_pair(
                model_obj, prep_fn, ds, family_cfg, year, year, scaler_obj, rbf_sampler=rbf_sampler, default_threshold=default_threshold
            )
            result = evaluate_one_pair(
                model_obj, prep_fn, ds, family_cfg, year, year, scaler_obj, threshold, threshold_meta, rbf_sampler=rbf_sampler
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'own_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_prior_years():
        rows = []
        for model_year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue
            for eval_year in [y for y in years if y <= model_year]:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

                threshold, threshold_meta = compute_threshold_for_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    rbf_sampler=rbf_sampler,
                    default_threshold=default_threshold,
                )
                result = evaluate_one_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    threshold,
                    threshold_meta,
                    rbf_sampler=rbf_sampler,
                )
                if result is None:
                    continue
                result.update({
                    'family_id': family_id,
                    'family_label': family_cfg['label'],
                    'table_type': 'prior_years',
                    'scaler_year_used': int(scaler_year),
                    'scaler_policy_used': scaler_policy,
                })
                rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_next_year():
        rows = []
        for model_year in years:
            candidate = model_year + 1
            if candidate not in years:
                continue
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue

            scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, candidate)
            scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

            threshold, threshold_meta = compute_threshold_for_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                model_year,
                candidate,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
            )
            result = evaluate_one_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                model_year,
                candidate,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'next_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    def _run_table_cumulative():
        rows = []
        year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
        for model_year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
            if model_obj is None:
                continue

            X_blocks = []
            y_blocks = []
            scaler_policies = []
            scaler_years = []
            threshold_rows = []

            for eval_year in [y for y in years if y <= model_year]:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)
                if scaler_obj is None and family_cfg['scaler_template'] is not None:
                    continue

                eval_year_idx = year_to_idx[int(eval_year)]
                scale_mode = 'none' if scaler_obj is None else 'transform'
                X_block, y_block, _ = prep_fn(ds, test_pixel_indices, eval_year_idx, scaler=scaler_obj, scaler_mode=scale_mode)
                if len(X_block) == 0:
                    continue

                validate_scaler_features(X_block, scaler_obj, family_cfg['label'], model_year, eval_year, 'cumulative_test_block')

                if rbf_sampler is not None:
                    X_block = rbf_sampler.transform(X_block)

                validate_model_features(X_block, model_obj, family_cfg['label'], model_year, eval_year, 'cumulative_test_block')

                X_blocks.append(X_block)
                y_blocks.append(y_block)
                scaler_policies.append(scaler_policy)
                scaler_years.append(int(scaler_year))

                threshold, threshold_meta = compute_threshold_for_pair(
                    model_obj,
                    prep_fn,
                    ds,
                    family_cfg,
                    model_year,
                    eval_year,
                    scaler_obj,
                    rbf_sampler=rbf_sampler,
                    default_threshold=default_threshold,
                )
                threshold_rows.append((threshold, threshold_meta))

            if not X_blocks:
                continue

            threshold, threshold_meta = threshold_rows[-1]
            X_test = np.vstack(X_blocks)
            y_test = np.concatenate(y_blocks)

            validate_model_features(X_test, model_obj, family_cfg['label'], model_year, model_year, 'cumulative_test_stack')

            y_proba = model_obj.predict_proba(X_test)[:, 1]
            y_pred = (y_proba >= threshold).astype(int)
            metrics = compute_metrics(y_test, y_pred, y_proba)

            rows.append({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'cumulative',
                'model_year': int(model_year),
                'eval_year': int(model_year),
                'n_test_samples': int(len(y_test)),
                'years_included': ','.join(str(y) for y in years if y <= model_year),
                'threshold_used': float(threshold),
                'threshold_source': threshold_meta.get('source', 'unknown'),
                'threshold_reason': threshold_meta.get('reason', ''),
                'n_val_threshold_samples': int(threshold_meta.get('n_val_samples', 0)),
                'scaler_year_used': int(scaler_years[-1]),
                'scaler_policy_used': scaler_policies[-1],
                **metrics,
            })
        return pd.DataFrame(rows)

    def _run_table_final_model_each_year():
        final_model = None
        final_model_path = family_cfg.get('final_model_path')
        if final_model_path and final_model_path.exists():
            with open(final_model_path, 'rb') as f:
                final_model = pickle.load(f)
        else:
            return pd.DataFrame([])

        final_scaler = load_final_scaler(family_cfg)
        rows = []
        for eval_year in years:
            if final_scaler is not None:
                scaler_obj = final_scaler
                scaler_year = max(years)
                scaler_policy = 'family_final_scaler'
            else:
                scaler_year, scaler_policy = resolve_scaler_year(family_cfg, max(years), eval_year)
                scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)
            threshold, threshold_meta = compute_threshold_for_pair(
                final_model,
                prep_fn,
                ds,
                family_cfg,
                max(years),
                eval_year,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
            )
            result = evaluate_one_pair(
                final_model,
                prep_fn,
                ds,
                family_cfg,
                max(years),
                eval_year,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'final_model_each_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    table_runners = {
        'own_year': _run_table_own_year,
        'prior_years': _run_table_prior_years,
        'next_year': _run_table_next_year,
        'cumulative': _run_table_cumulative,
        'final_model_each_year': _run_table_final_model_each_year,
    }

    table_frames = {}
    for table_name, table_fn in table_runners.items():
        table_path = family_out_dir / f'{family_id}_{table_name}.csv'
        df_table, skipped = load_or_run_table(table_path, table_fn)
        table_frames[table_name] = df_table
        status_rows.append({
            'family_id': family_id,
            'table_name': table_name,
            'skipped_existing': bool(skipped),
            'row_count': int(len(df_table)),
            'csv_path': str(table_path),
        })

    combined = pd.concat([df for df in table_frames.values() if len(df) > 0], ignore_index=True) if table_frames else pd.DataFrame([])
    combined_path = family_out_dir / f'{family_id}_combined.csv'
    save_table(combined, combined_path)

    status_df = pd.DataFrame(status_rows)
    save_table(status_df, family_out_dir / f'{family_id}_status.csv')

    return table_frames, combined, status_df


def run_all_families(default_threshold=0.5):
    all_family_combined = []
    all_status = []

    for family_id, family_cfg in FAMILIES.items():
        print('\n' + '=' * 100)
        print(f"FAMILY: {family_id} | {family_cfg['label']}")
        print('=' * 100)
        try:
            _, family_combined_df, family_status_df = evaluate_family(
                family_id=family_id,
                family_cfg=family_cfg,
                default_threshold=default_threshold,
            )
            if len(family_combined_df) > 0:
                all_family_combined.append(family_combined_df)
            all_status.append(family_status_df)
        except Exception as exc:
            all_status.append(pd.DataFrame([{
                'family_id': family_id,
                'table_name': '__family_failure__',
                'skipped_existing': False,
                'row_count': 0,
                'csv_path': '',
                'error': str(exc),
            }]))
            print(f'ERROR in family {family_id}: {exc}')

    global_combined_df = pd.concat(all_family_combined, ignore_index=True) if all_family_combined else pd.DataFrame([])
    global_status_df = pd.concat(all_status, ignore_index=True) if all_status else pd.DataFrame([])

    save_table(global_combined_df, EVAL_DIR / 'all_families_combined.csv')
    save_table(global_status_df, EVAL_DIR / 'all_families_status.csv')

    print('\n' + '=' * 100)
    print('GLOBAL SUMMARY')
    print('=' * 100)
    print(f'Combined rows: {len(global_combined_df):,}')
    print(f'Status rows:   {len(global_status_df):,}')

    return global_combined_df, global_status_df


# Run this cell when you are ready for full evaluation.
global_combined_results_df, global_status_df = run_all_families(default_threshold=0.5)

print('Pipeline implementation complete. Uncomment the last line to execute full evaluation.')


FAMILY: baseline | SGD Baseline
SKIP existing table: baseline_own_year.csv
SKIP existing table: baseline_prior_years.csv
SKIP existing table: baseline_next_year.csv
SKIP existing table: baseline_cumulative.csv
SKIP existing table: baseline_final_model_each_year.csv
ERROR in family baseline: No columns to parse from file

FAMILY: huber | SGD Huber


C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)


ERROR in family huber: X has 22 features, but StandardScaler is expecting 14 features as input.

FAMILY: lagged_features | SGD Lagged Features
SKIP existing table: lagged_features_own_year.csv
SKIP existing table: lagged_features_prior_years.csv
SKIP existing table: lagged_features_next_year.csv
SKIP existing table: lagged_features_cumulative.csv
SKIP existing table: lagged_features_final_model_each_year.csv
ERROR in family lagged_features: No columns to parse from file

FAMILY: lagged_features_incremental_scaler | SGD Lagged Features (Incremental Scaler)
SKIP existing table: lagged_features_incremental_scaler_own_year.csv
SKIP existing table: lagged_features_incremental_scaler_prior_years.csv
SKIP existing table: lagged_features_incremental_scaler_next_year.csv
SKIP existing table: lagged_features_incremental_scaler_cumulative.csv
SKIP existing table: lagged_features_incremental_scaler_final_model_each_year.csv
ERROR in family lagged_features_incremental_scaler: No columns to parse fr

C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\52

SAVED table: prevyears_monthly_features_experience_replay_own_year.csv (6 rows)


...

SAVED table: mlp_prevyears_monthly_features_incremental_scaler_next_year.csv (5 rows)


C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\52

SAVED table: mlp_prevyears_monthly_features_incremental_scaler_cumulative.csv (6 rows)


C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\52

SAVED table: mlp_prevyears_monthly_features_incremental_scaler_final_model_each_year.csv (6 rows)

GLOBAL SUMMARY
Combined rows: 220
Status rows:   31
Pipeline implementation complete. Uncomment the last line to execute full evaluation.


In [4]:
# Section: Table-structured metric matrices (years as columns)
# F1 and PR AUC are displayed as separate tables.

summary_root = EVAL_DIR / 'summary_matrices'
summary_root.mkdir(parents=True, exist_ok=True)

# Remove legacy single prior-years exports to enforce per-model-year output format.
legacy_prior_paths = [
    summary_root / 'metric_matrix_prior_years_f1.csv',
    summary_root / 'metric_matrix_prior_years_pr_auc.csv',
]
for legacy_path in legacy_prior_paths:
    if legacy_path.exists():
        legacy_path.unlink()
        print(f'Removed legacy export: {legacy_path.name}')

required_cols = ['family_id', 'table_type', 'eval_year', 'f1_score', 'pr_auc']
all_frames = []
skipped_families = []

for family_id, family_cfg in FAMILIES.items():
    family_dir = EVAL_DIR / family_id
    combined_csv = family_dir / f'{family_id}_combined.csv'

    df = None
    reason = None

    if combined_csv.exists():
        try:
            df = pd.read_csv(combined_csv)
        except Exception as exc:
            reason = f'failed reading combined csv ({exc})'
    else:
        fallback_files = sorted(family_dir.glob(f'{family_id}_*.csv')) if family_dir.exists() else []
        fallback_files = [p for p in fallback_files if not p.name.endswith('_status.csv') and not p.name.endswith('_combined.csv')]
        if fallback_files:
            frames = []
            for file_path in fallback_files:
                try:
                    frames.append(pd.read_csv(file_path))
                except Exception:
                    pass
            if frames:
                df = pd.concat(frames, ignore_index=True)
            else:
                reason = 'fallback table csv files could not be read'
        else:
            reason = 'combined csv not found'

    if df is None or df.empty:
        skipped_families.append((family_id, reason or 'empty dataframe'))
        continue

    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        skipped_families.append((family_id, f'missing columns: {missing_required}'))
        continue

    # model_year is required for per-model-year prior tables; keep NaN for table types where unavailable.
    if 'model_year' not in df.columns:
        df['model_year'] = np.nan

    df = df[required_cols + ['model_year']].copy()
    df['family_id'] = df['family_id'].fillna(family_id).astype(str)
    df['table_type'] = df['table_type'].astype(str)
    df['eval_year'] = pd.to_numeric(df['eval_year'], errors='coerce')
    df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')
    df['f1_score'] = pd.to_numeric(df['f1_score'], errors='coerce')
    df['pr_auc'] = pd.to_numeric(df['pr_auc'], errors='coerce')
    df = df.dropna(subset=['eval_year'])
    all_frames.append(df)

if not all_frames:
    print('No usable evaluation outputs found for matrix generation.')
else:
    results_df = pd.concat(all_frames, ignore_index=True)

    # Keep standard matrix exports for non-prior table types.
    table_types = [
        'own_year',
        'next_year',
        'cumulative',
        'final_model_each_year',
    ]

    metric_specs = [
        ('f1_score', 'f1'),
        ('pr_auc', 'pr_auc'),
    ]

    summary_outputs = {}

    for table_type in table_types:
        table_slice = results_df[results_df['table_type'] == table_type].copy()
        if table_slice.empty:
            print(f'[{table_type}] No rows available.')
            continue

        print('\n' + '=' * 100)
        print(f'TABLE: {table_type}')
        print('Rows are family_id. Columns are evaluation years.')
        print('Duplicate family-year rows are reduced using mean across contributing rows.')
        print('=' * 100)

        for metric_col, metric_label in metric_specs:
            metric_df = table_slice[['family_id', 'eval_year', metric_col]].copy()
            metric_df = metric_df.dropna(subset=[metric_col])

            if metric_df.empty:
                print(f'[{table_type}] {metric_label}: no non-null values available.')
                continue

            matrix_df = (
                metric_df
                .groupby(['family_id', 'eval_year'], as_index=False)[metric_col]
                .mean()
                .pivot(index='family_id', columns='eval_year', values=metric_col)
                .sort_index(axis=0)
                .sort_index(axis=1)
            )

            matrix_df.columns = [str(int(c)) if pd.notna(c) else str(c) for c in matrix_df.columns]
            matrix_df = matrix_df.reset_index().rename(columns={'family_id': 'row'})

            summary_outputs[(table_type, metric_label)] = matrix_df

            export_path = summary_root / f'metric_matrix_{table_type}_{metric_label}.csv'
            matrix_df.to_csv(export_path, index=False)

            print(f'\n{metric_label.upper()} matrix')
            display(matrix_df)
            print(f'Saved: {export_path}')

    # Dedicated end section: one prior-years table per model year (columns grow with model year).
    prior_slice = results_df[results_df['table_type'] == 'prior_years'].copy()

    print('\n' + '=' * 100)
    print('TABLE SECTION: prior_years (per model year)')
    print('Each table is one model year. Rows are family_id. Columns are eval_year <= model_year.')
    print('=' * 100)

    if prior_slice.empty:
        print('[prior_years] No rows available.')
    elif prior_slice['model_year'].dropna().empty:
        print('[prior_years] Missing model_year values; cannot build per-model-year tables.')
    else:
        model_years = sorted(prior_slice['model_year'].dropna().astype(int).unique())

        for model_year in model_years:
            model_slice = prior_slice[prior_slice['model_year'].astype('Int64') == model_year].copy()
            if model_slice.empty:
                continue

            print('\n' + '-' * 100)
            print(f'PRIOR MODEL YEAR: {model_year}')
            print('-' * 100)

            for metric_col, metric_label in metric_specs:
                metric_df = model_slice[['family_id', 'eval_year', metric_col]].copy()
                metric_df = metric_df.dropna(subset=[metric_col])

                if metric_df.empty:
                    print(f'[prior_years_model_{model_year}] {metric_label}: no non-null values available.')
                    continue

                matrix_df = (
                    metric_df
                    .groupby(['family_id', 'eval_year'], as_index=False)[metric_col]
                    .mean()
                    .pivot(index='family_id', columns='eval_year', values=metric_col)
                    .sort_index(axis=0)
                    .sort_index(axis=1)
                )

                matrix_df.columns = [str(int(c)) if pd.notna(c) else str(c) for c in matrix_df.columns]
                matrix_df = matrix_df.reset_index().rename(columns={'family_id': 'row'})

                table_key = f'prior_years_model_{model_year}'
                summary_outputs[(table_key, metric_label)] = matrix_df

                export_path = summary_root / f'metric_matrix_{table_key}_{metric_label}.csv'
                matrix_df.to_csv(export_path, index=False)

                print(f'\n{metric_label.upper()} matrix')
                display(matrix_df)
                print(f'Saved: {export_path}')

    if skipped_families:
        print('\nSkipped families (missing/invalid outputs):')
        for family_name, reason in skipped_families:
            print(f' - {family_name}: {reason}')

    overall_summary = pd.DataFrame([
        {
            'table_type': table_type,
            'metric': metric_label,
            'row_count': len(df_out),
        }
        for (table_type, metric_label), df_out in summary_outputs.items()
    ])

    overall_export_path = summary_root / 'metric_matrix_overview.csv'
    overall_summary.to_csv(overall_export_path, index=False)

    print('\nSummary matrix exports complete.')
    print(f'Output folder: {summary_root}')
    display(overall_summary.sort_values(['table_type', 'metric']).reset_index(drop=True))


TABLE: own_year
Rows are family_id. Columns are evaluation years.
Duplicate family-year rows are reduced using mean across contributing rows.

F1 matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.181211,0.394452,0.218509,0.256754,0.230862,0.258414
1,lagged_features,0.183724,0.394536,0.393827,0.419739,0.408259,0.413103
2,lagged_features_incremental_scaler,0.199615,0.395409,0.394097,0.412271,0.401910,0.411766
3,mlp_prevyears_monthly_features_incremental_scaler,0.266705,0.450551,0.470878,0.484801,0.474810,0.491599
4,neighbours,0.187932,0.395161,0.185108,0.214480,0.198482,0.231862
5,prevyears_monthly_features,0.180581,0.405588,0.417279,0.446585,0.407614,0.415744
6,prevyears_monthly_features_experience_replay,0.187821,0.409010,0.409905,0.439004,0.404064,0.416170
7,prevyears_monthly_features_incremental_scaler,0.192974,0.412624,0.411193,0.437638,0.413210,0.420131
8,prevyears_monthly_features_incremental_scaler_...,0.186693,0.396712,0.401195,0.431044,0.390829,0.399650
9,prevyears_monthly_features_rbf,0.035619,0.043850,0.040473,0.053503,0.033240,0.061141


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_own_year_f1.csv

PR_AUC matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.102965,0.379073,0.160839,0.193445,0.163037,0.195861
1,lagged_features,0.109286,0.378105,0.343162,0.385600,0.395786,0.386227
2,lagged_features_incremental_scaler,0.121508,0.375879,0.341089,0.376934,0.386522,0.386627
3,mlp_prevyears_monthly_features_incremental_scaler,0.195739,0.432104,0.431288,0.453995,0.444383,0.468410
4,neighbours,0.111358,0.383249,0.120917,0.152466,0.128176,0.153380
5,prevyears_monthly_features,0.105772,0.389604,0.363116,0.412449,0.382962,0.392930
6,prevyears_monthly_features_experience_replay,0.104466,0.395264,0.355973,0.403962,0.385958,0.397788
7,prevyears_monthly_features_incremental_scaler,0.112571,0.395498,0.359045,0.403619,0.391261,0.401422
8,prevyears_monthly_features_incremental_scaler_...,0.105441,0.381160,0.343636,0.388120,0.358327,0.376471
9,prevyears_monthly_features_rbf,0.017331,0.019169,0.019136,0.029920,0.016893,0.029462


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_own_year_pr_auc.csv

TABLE: next_year
Rows are family_id. Columns are evaluation years.
Duplicate family-year rows are reduced using mean across contributing rows.

F1 matrix


...

,row,2017,2018,2019,2020,2021,2022
0,baseline,0.176663,0.236311,0.204680,0.221384,0.207831,0.258414
1,lagged_features,0.158352,0.363753,0.389842,0.389340,0.399286,0.413103
2,lagged_features_incremental_scaler,0.157600,0.359747,0.390527,0.402285,0.386724,0.411766
3,mlp_prevyears_monthly_features_incremental_scaler,0.169306,0.354291,0.453030,0.489709,0.458633,0.491599
4,neighbours,0.166848,0.230557,0.174148,0.183790,0.172171,0.231862
5,prevyears_monthly_features,0.167729,0.376482,0.401107,0.421222,0.405427,0.415744
6,prevyears_monthly_features_experience_replay,0.162581,0.346798,0.396538,0.425784,0.399305,0.416170
7,prevyears_monthly_features_incremental_scaler,0.162130,0.348546,0.398417,0.430781,0.397085,0.420131
8,prevyears_monthly_features_incremental_scaler_...,0.179674,0.364868,0.386105,0.410012,0.385430,0.399650
9,prevyears_monthly_features_rbf,0.035629,0.040689,0.040496,0.051361,0.033256,0.061141


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_prior_years_model_2022_f1.csv

PR_AUC matrix


,row,2017,2018,2019,2020,2021,2022
0,baseline,0.114295,0.190124,0.148612,0.163078,0.147342,0.195861
1,lagged_features,0.086834,0.336067,0.337247,0.353612,0.372492,0.386227
2,lagged_features_incremental_scaler,0.085307,0.332058,0.339775,0.368690,0.368186,0.386627
3,mlp_prevyears_monthly_features_incremental_scaler,0.096987,0.312553,0.402358,0.454438,0.429059,0.468410
4,neighbours,0.103237,0.170805,0.109106,0.120809,0.101794,0.153380
5,prevyears_monthly_features,0.092160,0.356850,0.356100,0.392861,0.383586,0.392930
6,prevyears_monthly_features_experience_replay,0.085825,0.332732,0.351281,0.396565,0.372886,0.397788
7,prevyears_monthly_features_incremental_scaler,0.084545,0.332065,0.354247,0.403411,0.370796,0.401422
8,prevyears_monthly_features_incremental_scaler_...,0.105397,0.342351,0.339520,0.375780,0.358094,0.376471
9,prevyears_monthly_features_rbf,0.016984,0.019534,0.018757,0.029362,0.016832,0.029462


Saved: eval_outputs\unified_eval\summary_matrices\metric_matrix_prior_years_model_2022_pr_auc.csv

Skipped families (missing/invalid outputs):
 - huber: combined csv not found

Summary matrix exports complete.
Output folder: eval_outputs\unified_eval\summary_matrices


,table_type,metric,row_count
0,cumulative,f1,10
1,cumulative,pr_auc,10
2,final_model_each_year,f1,5
3,final_model_each_year,pr_auc,5
4,next_year,f1,10
5,next_year,pr_auc,10
6,own_year,f1,10
7,own_year,pr_auc,10
8,prior_years_model_2017,f1,10
9,prior_years_model_2017,pr_auc,10


## Debugging: In-Memory Single-Family Evaluation

This section runs the evaluation pipeline for only the prevyears_monthly_features family and generates all tables without saving any files.

In [5]:
# Debug run for one family only, in-memory only (no save_table calls)

def debug_run_single_family_no_save(family_id='prevyears_monthly_features', default_threshold=0.5):
    if family_id not in FAMILIES:
        raise KeyError(f'Unknown family_id: {family_id}')

    family_cfg = FAMILIES[family_id]
    prep_fn = PREP_FN_BY_KIND[family_cfg['prep_kind']]
    ds = load_family_dataset(family_cfg)
    models_dir = family_cfg['models_dir']

    if not models_dir.exists():
        raise FileNotFoundError(f'Model directory missing: {models_dir.resolve()}')

    years = infer_years_from_models(models_dir, family_cfg['model_template'])
    if not years:
        raise RuntimeError(f'No model years found for {family_id}')

    rbf_sampler = None
    if 'rbf_sampler_path' in family_cfg:
        rbf_path = family_cfg['rbf_sampler_path']
        if rbf_path.exists():
            with open(rbf_path, 'rb') as f:
                rbf_sampler = pickle.load(f)

    def run_table_own_year():
        rows = []
        for year in years:
            model_obj = load_model(models_dir, family_cfg['model_template'], year)
            if model_obj is None:
                continue

            scaler_year, scaler_policy = resolve_scaler_year(family_cfg, year, year)
            scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)

            threshold, threshold_meta = compute_threshold_for_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                year,
                year,
                scaler_obj,
                rbf_sampler=rbf_sampler,
                default_threshold=default_threshold,
            )
            result = evaluate_one_pair(
                model_obj,
                prep_fn,
                ds,
                family_cfg,
                year,
                year,
                scaler_obj,
                threshold,
                threshold_meta,
                rbf_sampler=rbf_sampler,
            )
            if result is None:
                continue
            result.update({
                'family_id': family_id,
                'family_label': family_cfg['label'],
                'table_type': 'own_year',
                'scaler_year_used': int(scaler_year),
                'scaler_policy_used': scaler_policy,
            })
            rows.append(result)
        return pd.DataFrame(rows)

    own_year_df = run_table_own_year()
    return own_year_df


# Execute debug run for only prevyears_monthly_features.
# This computes only the own_year table in-memory and does not write any output files.
debug_own_year_prevyears_monthly_features = debug_run_single_family_no_save(
    family_id='prevyears_monthly_features',
    default_threshold=0.5,
)

print('Debug single-family run complete: prevyears_monthly_features')
print(f'  own_year: {len(debug_own_year_prevyears_monthly_features):,} rows')

print('\n' + '=' * 100)
print('DEBUG TABLE: own_year')
print('=' * 100)
display(debug_own_year_prevyears_monthly_features)


C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\52

Debug single-family run complete: prevyears_monthly_features
  own_year: 6 rows

DEBUG TABLE: own_year


,model_year,eval_year,n_test_samples,threshold_used,threshold_source,threshold_reason,n_val_threshold_samples,accuracy,precision,recall,f1_score,roc_auc,pr_auc,family_id,family_label,table_type,scaler_year_used,scaler_policy_used
0,2017,2017,1278700,0.833363,computed_pr_f1,max F1 on validation PR curve,1270825,0.934193,0.116614,0.399983,0.180581,0.833684,0.105772,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2017,evaluation_year_scaler
1,2018,2018,1281398,0.906836,computed_pr_f1,max F1 on validation PR curve,1271706,0.975694,0.448847,0.369935,0.405588,0.896495,0.389604,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2018,evaluation_year_scaler
2,2019,2019,1281008,0.875020,computed_pr_f1,max F1 on validation PR curve,1271570,0.977454,0.447256,0.391068,0.417279,0.867258,0.363116,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2019,evaluation_year_scaler
3,2020,2020,1281489,0.895329,computed_pr_f1,max F1 on validation PR curve,1271629,0.975653,0.503903,0.400975,0.446585,0.895978,0.412449,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2020,evaluation_year_scaler
4,2021,2021,1281313,0.930962,computed_pr_f1,max F1 on validation PR curve,1271639,0.983533,0.519539,0.335366,0.407614,0.886329,0.382962,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2021,evaluation_year_scaler
5,2022,2022,1282452,0.912834,computed_pr_f1,max F1 on validation PR curve,1272609,0.960923,0.393260,0.440955,0.415744,0.881965,0.392930,prevyears_monthly_features,SGD Prevyears + Monthly,own_year,2022,evaluation_year_scaler


In [6]:
# Debugging cell: inspect why threshold selection picks extreme values (e.g., ~1.0)


def _prepare_features_common_training_order_fix(
    ds,
    pixel_indices,
    year_idx,
    scaler=None,
    scaler_mode='auto',
    include_last_year=True,
    include_monthly=False,
    include_neighbourhood=False,
):
    """
    Local debug-only feature builder that matches the original training notebook's
    feature order exactly for prevyears/monthly families.
    """
    if year_idx == 0:
        return np.empty((0, 0)), np.empty((0,)), scaler

    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f'Invalid scaler_mode: {scaler_mode}')

    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    # Keep base order aligned with training notebook.
    features_list = [s2_features, dem_features, ndvi_features, ndwi_features]

    if include_last_year:
        ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
        ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
        ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))
        features_list.extend([ndvi_last_year, ndwi_last_year])

        required_last_year_bands = ['B04', 'B03', 'B06']
        band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
        missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
        if missing_bands:
            raise ValueError(f'Missing required S2 bands for last-year features: {missing_bands}')
        for band in required_last_year_bands:
            band_values = ds_prev['s2_bands'].sel(s2_band=band).values
            band_values = np.where(np.isnan(band_values), 0, band_values)
            features_list.append(band_values.reshape(-1, 1))

    # IMPORTANT: NBR is appended after last-year features in the training notebook.
    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if include_neighbourhood:
        for var_name in ['ndvi_neighbour', 'ndwi_neighbour', 's2_b03_neighbour', 's2_b04_neighbour']:
            if var_name in ds.data_vars:
                features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    if include_monthly:
        yearly_index_feature_names = [
            'ndvi_cv_year',
            'ndvi_max_m2m_drop_year',
            'ndvi_max_year',
            'ndvi_min_year',
            'ndvi_std_year',
            'ndwi_cv_year',
            'ndwi_max_m2m_drop_year',
            'ndwi_max_year',
            'ndwi_min_year',
            'ndwi_std_year',
        ]
        for feature_name in yearly_index_feature_names:
            if feature_name in ds.data_vars:
                features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X = X[nan_mask]
    y = y[nan_mask]

    if len(X) == 0:
        return np.empty((0, 0)), np.empty((0,)), scaler

    if scaler_mode == 'none':
        return X, y, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'n_features_in_'):
            X = scaler.transform(X)
        else:
            X = scaler.fit_transform(X)
    elif scaler_mode == 'fit':
        X = scaler.fit_transform(X)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X)
        X = scaler.transform(X)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'n_features_in_'):
            raise ValueError("Scaler must be fitted before scaler_mode='transform'.")
        X = scaler.transform(X)

    return X, y, scaler


def _debug_prep_fn_with_order_fix(family_cfg):
    prep_kind = family_cfg['prep_kind']

    def _prep(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
        return _prepare_features_common_training_order_fix(
            ds=ds,
            pixel_indices=pixel_indices,
            year_idx=year_idx,
            scaler=scaler,
            scaler_mode=scaler_mode,
            include_last_year=(prep_kind != 'neighbourhood'),
            include_monthly=(prep_kind == 'monthly'),
            include_neighbourhood=(prep_kind == 'neighbourhood'),
        )

    return _prep


def debug_threshold_selection(
    family_id='prevyears_monthly_features_incremental_scaler',
    table_type='own_year',
    max_samples=300000,
    random_state=42,
    show_top_k=10,
    use_feature_order_fix=False,
):
    if family_id not in FAMILIES:
        raise KeyError(f'Unknown family_id: {family_id}')

    family_cfg = FAMILIES[family_id]
    prep_fn = PREP_FN_BY_KIND[family_cfg['prep_kind']]
    if use_feature_order_fix:
        prep_fn = _debug_prep_fn_with_order_fix(family_cfg)

    ds = load_family_dataset(family_cfg)
    models_dir = family_cfg['models_dir']

    years = infer_years_from_models(models_dir, family_cfg['model_template'])
    if not years:
        raise RuntimeError(f'No model years found for {family_id}')

    rbf_sampler = None
    if 'rbf_sampler_path' in family_cfg:
        rbf_path = family_cfg['rbf_sampler_path']
        if rbf_path.exists():
            with open(rbf_path, 'rb') as f:
                rbf_sampler = pickle.load(f)

    year_pairs = []
    if table_type == 'own_year':
        year_pairs = [(y, y) for y in years]
    elif table_type == 'prior_years':
        for model_year in years:
            for eval_year in [y for y in years if y <= model_year]:
                year_pairs.append((model_year, eval_year))
    else:
        raise ValueError("table_type must be 'own_year' or 'prior_years'")

    rng = np.random.default_rng(random_state)
    year_to_idx = {int(y): i for i, y in enumerate(ds.year.values)}
    summary_rows = []
    detailed = {}

    for model_year, eval_year in year_pairs:
        model_obj = load_model(models_dir, family_cfg['model_template'], model_year)
        if model_obj is None:
            continue

        scaler_year, scaler_policy = resolve_scaler_year(family_cfg, model_year, eval_year)
        scaler_obj = load_scaler(models_dir, family_cfg['scaler_template'], scaler_year)
        val_scale_mode = 'none' if scaler_obj is None else 'transform'

        eval_year_idx = year_to_idx[int(eval_year)]
        X_val, y_val, _ = prep_fn(
            ds,
            val_pixel_indices,
            eval_year_idx,
            scaler=scaler_obj,
            scaler_mode=val_scale_mode,
        )

        if len(X_val) == 0:
            continue

        if rbf_sampler is not None:
            X_val = rbf_sampler.transform(X_val)

        # Optional downsampling for fast diagnostics while preserving behavior trends.
        n_before = len(y_val)
        if max_samples is not None and len(y_val) > int(max_samples):
            sample_idx = rng.choice(len(y_val), size=int(max_samples), replace=False)
            X_val = X_val[sample_idx]
            y_val = y_val[sample_idx]

        y_score = model_obj.predict_proba(X_val)[:, 1]

        threshold, meta = best_f1_threshold(y_val, y_score, default_threshold=0.5)

        precision_vals, recall_vals, thresholds = precision_recall_curve(y_val, y_score)
        if len(thresholds) > 0:
            f1_vals = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (precision_vals[:-1] + recall_vals[:-1] + 1e-12)
            cand_df = pd.DataFrame({
                'threshold': thresholds.astype(float),
                'precision': precision_vals[:-1].astype(float),
                'recall': recall_vals[:-1].astype(float),
                'f1': f1_vals.astype(float),
            })
            cand_df = cand_df.sort_values('f1', ascending=False).reset_index(drop=True)
            top_candidates = cand_df.head(show_top_k)
        else:
            top_candidates = pd.DataFrame(columns=['threshold', 'precision', 'recall', 'f1'])

        q = np.quantile(y_score, [0.0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1.0])
        pos_rate = float(np.mean(y_val == 1)) if len(y_val) > 0 else np.nan

        summary_rows.append({
            'family_id': family_id,
            'model_year': int(model_year),
            'eval_year': int(eval_year),
            'feature_order_mode': 'training_order_fix' if use_feature_order_fix else 'current_eval_order',
            'n_val_samples_before_sampling': int(n_before),
            'n_val_samples_used': int(len(y_val)),
            'positive_rate': pos_rate,
            'threshold_selected': float(threshold),
            'threshold_source': meta.get('source', 'unknown'),
            'threshold_reason': meta.get('reason', ''),
            'score_min': float(np.min(y_score)),
            'score_p01': float(q[1]),
            'score_p05': float(q[2]),
            'score_p25': float(q[3]),
            'score_p50': float(q[4]),
            'score_p75': float(q[5]),
            'score_p95': float(q[6]),
            'score_p99': float(q[7]),
            'score_max': float(q[8]),
            'share_score_ge_0_999': float(np.mean(y_score >= 0.999)),
            'share_score_ge_0_9999': float(np.mean(y_score >= 0.9999)),
            'share_score_eq_1': float(np.mean(y_score == 1.0)),
            'share_score_le_0_001': float(np.mean(y_score <= 0.001)),
            'unique_score_count': int(np.unique(y_score).size),
            'scaler_year_used': int(scaler_year),
            'scaler_policy_used': scaler_policy,
        })

        detailed[(int(model_year), int(eval_year))] = {
            'top_f1_candidates': top_candidates,
            'y_score_sample': y_score[: min(20, len(y_score))],
        }

    summary_df = pd.DataFrame(summary_rows).sort_values(['model_year', 'eval_year']).reset_index(drop=True)
    return summary_df, detailed


# A/B debug run: current evaluation order vs training-order fix.
target_family_id = 'prevyears_monthly_features_incremental_scaler'

threshold_debug_summary_df, threshold_debug_detail = debug_threshold_selection(
    family_id=target_family_id,
    table_type='own_year',
    max_samples=300000,
    random_state=42,
    show_top_k=10,
    use_feature_order_fix=False,
)

threshold_debug_summary_df_fix, threshold_debug_detail_fix = debug_threshold_selection(
    family_id=target_family_id,
    table_type='own_year',
    max_samples=300000,
    random_state=42,
    show_top_k=10,
    use_feature_order_fix=True,
)

print(f'Threshold diagnostics complete (A/B comparison): {target_family_id}')

compare_cols = [
    'model_year',
    'eval_year',
    'feature_order_mode',
    'threshold_selected',
    'f1_proxy',
    'score_p95',
    'score_p99',
    'score_max',
    'share_score_ge_0_999',
    'share_score_eq_1',
    'unique_score_count',
]

# Add a compact proxy to compare where thresholds are landing.
for _df in [threshold_debug_summary_df, threshold_debug_summary_df_fix]:
    _df['f1_proxy'] = (2 * _df['positive_rate']) / (1 + _df['positive_rate'])

comparison_df = pd.concat([
    threshold_debug_summary_df[compare_cols],
    threshold_debug_summary_df_fix[compare_cols],
], ignore_index=True)

display(comparison_df.sort_values(['model_year', 'feature_order_mode']).reset_index(drop=True))

# Show top threshold candidates for the highest selected threshold in each mode.
for label, summary_df_local, detail_local in [
    ('CURRENT EVAL ORDER', threshold_debug_summary_df, threshold_debug_detail),
    ('TRAINING ORDER FIX', threshold_debug_summary_df_fix, threshold_debug_detail_fix),
]:
    if summary_df_local.empty:
        continue
    chosen_row = summary_df_local.sort_values('threshold_selected', ascending=False).iloc[0]
    chosen_key = (int(chosen_row['model_year']), int(chosen_row['eval_year']))
    print(f"\n[{label}] top F1 threshold candidates for model_year={chosen_key[0]}, eval_year={chosen_key[1]}")
    display(detail_local[chosen_key]['top_f1_candidates'])
    print('Sample predicted probabilities:')
    print(detail_local[chosen_key]['y_score_sample'])

C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\520865103.py:19: RuntimeWarning: Mean of empty slice
  s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)
C:\Users\bartu\AppData\Local\Temp\ipykernel_19900\34

Threshold diagnostics complete (A/B comparison): prevyears_monthly_features_incremental_scaler


,model_year,eval_year,feature_order_mode,threshold_selected,f1_proxy,score_p95,score_p99,score_max,share_score_ge_0_999,share_score_eq_1,unique_score_count
0,2017,2017,current_eval_order,0.882647,0.036540,0.817593,0.932315,0.999678,0.000010,0.000000,299997
1,2017,2017,training_order_fix,0.882647,0.036540,0.817593,0.932315,0.999678,0.000010,0.000000,299997
2,2018,2018,current_eval_order,0.931940,0.043244,0.781143,0.966062,0.999995,0.001347,0.000000,299995
3,2018,2018,training_order_fix,0.931940,0.043244,0.781143,0.966062,0.999995,0.001347,0.000000,299995
4,2019,2019,current_eval_order,0.881619,0.035981,0.703603,0.927093,1.000000,0.000833,0.000003,299999
5,2019,2019,training_order_fix,0.881619,0.035981,0.703603,0.927093,1.000000,0.000833,0.000003,299999
6,2020,2020,current_eval_order,0.890865,0.033838,0.712521,0.921614,1.000000,0.001073,0.000003,299998
7,2020,2020,training_order_fix,0.890865,0.033838,0.712521,0.921614,1.000000,0.001073,0.000003,299998
8,2021,2021,current_eval_order,0.931338,0.030869,0.666772,0.934892,1.000000,0.001383,0.000007,299996
9,2021,2021,training_order_fix,0.931338,0.030869,0.666772,0.934892,1.000000,0.001383,0.000007,299996



[CURRENT EVAL ORDER] top F1 threshold candidates for model_year=2018, eval_year=2018


,threshold,precision,recall,f1
0,0.931940,0.449933,0.355807,0.397372
1,0.931922,0.449847,0.355807,0.397339
2,0.931890,0.449762,0.355807,0.397305
3,0.931563,0.448728,0.356410,0.397276
4,0.931862,0.449676,0.355807,0.397272
5,0.932008,0.449914,0.355656,0.397271
6,0.931548,0.448642,0.356410,0.397243
7,0.931847,0.449590,0.355807,0.397238
8,0.931995,0.449828,0.355656,0.397237
9,0.931544,0.448557,0.356410,0.397210


Sample predicted probabilities:
[7.62546202e-04 4.38763282e-01 1.70020317e-01 1.41777610e-01
 8.76152905e-04 1.04295972e-01 1.44886492e-01 1.72388968e-01
 7.68371382e-01 3.36635868e-01 2.79408129e-01 2.76643400e-01
 1.28915213e-01 1.82928010e-01 2.31077871e-02 1.36051963e-01
 8.04992609e-02 3.38770460e-01 3.51220972e-01 7.49866687e-01]

[TRAINING ORDER FIX] top F1 threshold candidates for model_year=2018, eval_year=2018


,threshold,precision,recall,f1
0,0.931940,0.449933,0.355807,0.397372
1,0.931922,0.449847,0.355807,0.397339
2,0.931890,0.449762,0.355807,0.397305
3,0.931563,0.448728,0.356410,0.397276
4,0.931862,0.449676,0.355807,0.397272
5,0.932008,0.449914,0.355656,0.397271
6,0.931548,0.448642,0.356410,0.397243
7,0.931847,0.449590,0.355807,0.397238
8,0.931995,0.449828,0.355656,0.397237
9,0.931544,0.448557,0.356410,0.397210


Sample predicted probabilities:
[7.62546202e-04 4.38763282e-01 1.70020317e-01 1.41777610e-01
 8.76152905e-04 1.04295972e-01 1.44886492e-01 1.72388968e-01
 7.68371382e-01 3.36635868e-01 2.79408129e-01 2.76643400e-01
 1.28915213e-01 1.82928010e-01 2.31077871e-02 1.36051963e-01
 8.04992609e-02 3.38770460e-01 3.51220972e-01 7.49866687e-01]


In [7]:
# Baseline compatibility check for the notebook-imported feature prep.

baseline_cfg_check = FAMILIES['baseline']
baseline_ds_check = load_family_dataset(baseline_cfg_check)
baseline_years_check = infer_years_from_models(
    baseline_cfg_check['models_dir'],
    baseline_cfg_check['model_template'],
)

if not baseline_years_check:
    raise RuntimeError('No baseline model years found for compatibility check.')

baseline_year_check = int(baseline_years_check[0])
baseline_model_check = load_model(
    baseline_cfg_check['models_dir'],
    baseline_cfg_check['model_template'],
    baseline_year_check,
    )
baseline_scaler_check = load_scaler(
    baseline_cfg_check['models_dir'],
    baseline_cfg_check['scaler_template'],
    baseline_year_check,
    )

baseline_prep_fn_check = PREP_FN_BY_KIND[baseline_cfg_check['prep_kind']]
baseline_threshold_check, baseline_threshold_meta_check = compute_threshold_for_pair(
    baseline_model_check,
    baseline_prep_fn_check,
    baseline_ds_check,
    baseline_cfg_check,
    baseline_year_check,
    baseline_year_check,
    baseline_scaler_check,
    default_threshold=0.5,
    )
baseline_result_check = evaluate_one_pair(
    baseline_model_check,
    baseline_prep_fn_check,
    baseline_ds_check,
    baseline_cfg_check,
    baseline_year_check,
    baseline_year_check,
    baseline_scaler_check,
    baseline_threshold_check,
    baseline_threshold_meta_check,
    )

baseline_validation_summary = pd.DataFrame([
    {
        'reference_notebook': str(baseline_cfg_check['reference_notebook']),
        'model_year': baseline_year_check,
        'prep_kind': baseline_cfg_check['prep_kind'],
        'scaler_expected_features': getattr(baseline_scaler_check, 'n_features_in_', np.nan),
        'model_expected_features': getattr(baseline_model_check, 'n_features_in_', np.nan),
        'threshold_used': baseline_threshold_check,
        'n_test_samples': baseline_result_check['n_test_samples'],
        'f1_score': baseline_result_check['f1_score'],
        'pr_auc': baseline_result_check['pr_auc'],
    }
])

print('Baseline notebook-prep compatibility check passed.')
display(baseline_validation_summary)

<string>:27: RuntimeWarning: Mean of empty slice


  Dropped 101 samples with invalid labels (class 255) for year 1
  Dropped 1324 samples with NaN values (0.1%) for year 1


<string>:27: RuntimeWarning: Mean of empty slice


  Dropped 291 samples with invalid labels (class 255) for year 1
  Dropped 2087 samples with NaN values (0.2%) for year 1
Baseline notebook-prep compatibility check passed.


,reference_notebook,model_year,prep_kind,scaler_expected_features,model_expected_features,threshold_used,n_test_samples,f1_score,pr_auc
0,SGD Classifier.ipynb,2017,baseline_notebook,17,17,0.871201,1281614,0.181211,0.102965
